In [6]:
!pip install -q transformers datasets accelerate scikit-learn scipy

In [1]:
# Minimal Mamba-130M unlearning feasibility check (FAST VERSION)
# Memory & Speed optimized for pure PyTorch fallback on a Tesla T4

!pip install -q transformers datasets accelerate scikit-learn

import os, json, math, random, gc, time, inspect
import numpy as np
import torch

# Disable cuDNN to avoid the T4 conv1d backward crash
torch.backends.cudnn.enabled = False

import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import roc_auc_score

SEED = 42
MODEL = "state-spaces/mamba-130m-hf"
MAX_LEN = 64           # Reduced from 128 to speed up the pure PyTorch scan

# Fast feasibility parameters
FT_EPOCHS = 5
FT_MICRO_BS = 4
FT_ACCUM_STEPS = 4     # Effective batch size = 16
FT_LR = 1e-3           # Higher LR to force memorization quickly on the subset

# Unlearning params
UN_STEPS = 30
UN_MICRO_BS = 4
UN_ACCUM_STEPS = 4
UN_LRS = [1e-4, 3e-4]
METHODS = ["ga", "graddiff", "npo"]
NPO_BETA = 0.1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

set_seed()
print("device:", DEVICE, "| gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

# ---------- data (SLICED FOR FAST FEASIBILITY) ----------
def load_tofu(name):
    d = load_dataset("locuslab/TOFU", name)
    split = "train" if "train" in d else list(d.keys())[0]
    return [dict(q=r["question"], a=r["answer"]) for r in d[split]]

print("Loading and slicing dataset...")
# We only use a small subset to prove memorization/unlearning mechanics
forget = load_tofu("forget05")[:50]
retain95 = load_tofu("retain95")
holdout = retain95[-50:]
retain_train = retain95[:150]
print(f"splits: forget {len(forget)} | holdout {len(holdout)} | retain {len(retain_train)}")

# ---------- tokenizer / encoding ----------
tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token

def encode(rows):
    out = []
    for r in rows:
        p = tok(f"Question: {r['q']}\nAnswer:", add_special_tokens=False)["input_ids"]
        a = tok(" " + r["a"], add_special_tokens=False)["input_ids"] + [tok.eos_token_id]
        ids = (p + a)[:MAX_LEN]
        lab = ([-100] * len(p) + a)[:MAX_LEN]
        if len(p) >= len(ids): continue
        out.append(dict(ids=ids, labels=lab, q=r["q"], a=r["a"]))
    return out

forget_e = encode(forget); holdout_e = encode(holdout)
retain_e = encode(retain_train); target_e = retain_e + forget_e

def collate(batch):
    L = max(len(b["ids"]) for b in batch)
    ids = torch.full((len(batch), L), tok.pad_token_id, dtype=torch.long)
    lab = torch.full((len(batch), L), -100, dtype=torch.long)
    att = torch.zeros((len(batch), L), dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b["ids"]); ids[i, :n] = torch.tensor(b["ids"])
        lab[i, :n] = torch.tensor(b["labels"]); att[i, :n] = 1
    return ids.to(DEVICE), att.to(DEVICE), lab.to(DEVICE)

# ---------- model / loss ----------
def load_model(path=MODEL, for_training=False):
    m = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.float32, trust_remote_code=True)
    if for_training:
        m.gradient_checkpointing_enable()
        m.config.use_cache = False
    return m.to(DEVICE)

def token_nll(model, ids, att, lab):
    logits = model(input_ids=ids, attention_mask=att).logits[:, :-1].float()
    tgt = lab[:, 1:]
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1),
                           ignore_index=-100, reduction="none").view(tgt.shape), (tgt != -100)

def mean_nll(model, exs, bs=8):
    model.eval(); vals = []
    with torch.no_grad(), torch.autocast("cuda"):
        for i in range(0, len(exs), bs):
            nll, mask = token_nll(model, *collate(exs[i:i+bs]))
            vals.append((nll * mask).sum().item() / max(mask.sum().item(), 1))
    return float(np.mean(vals))

def eval_mia(model):
    fn, hn = mean_nll(model, forget_e), mean_nll(model, holdout_e)
    sf = np.array([-mean_nll(model, [e]) for e in forget_e])
    sh = np.array([-mean_nll(model, [e]) for e in holdout_e])
    y = np.r_[np.ones(len(sf)), np.zeros(len(sh))]
    return roc_auc_score(y, np.r_[sf, sh]), fn, hn

# ---------- fine-tune (target / oracle) ----------
def finetune(train_ex, tag):
    path = f"/content/{tag}_mamba"
    if os.path.exists(os.path.join(path, "config.json")): return path

    set_seed()
    model = load_model(for_training=True)
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=FT_LR, weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda")

    N = len(train_ex)
    spe = math.ceil(N / FT_MICRO_BS)

    for ep in range(FT_EPOCHS):
        perm = torch.randperm(N).tolist()
        run = 0.0
        opt.zero_grad(set_to_none=True)

        for bi in range(spe):
            batch = [train_ex[i] for i in perm[bi * FT_MICRO_BS : (bi + 1) * FT_MICRO_BS]]

            with torch.autocast("cuda"):
                nll, mask = token_nll(model, *collate(batch))
                loss = (nll * mask).sum() / mask.sum()
                loss = loss / FT_ACCUM_STEPS

            scaler.scale(loss).backward()

            if (bi + 1) % FT_ACCUM_STEPS == 0 or (bi + 1) == spe:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)

            run += loss.item() * FT_ACCUM_STEPS

        print(f"[{tag}] epoch {ep+1}/{FT_EPOCHS} loss {run/spe:.3f}")

    model.save_pretrained(path); tok.save_pretrained(path)
    del model, opt; gc.collect(); torch.cuda.empty_cache()
    return path

print("\n--- Fine-tuning Target (Mamba) ---")
target_path = finetune(target_e, "target")
print("\n--- Fine-tuning Oracle (Mamba) ---")
oracle_path = finetune(retain_e, "oracle")

# Evaluate baselines
for name, path in [("TARGET", target_path), ("ORACLE", oracle_path)]:
    m = load_model(path, for_training=False)
    mia, fn, hn = eval_mia(m)
    rn = mean_nll(m, retain_e)
    print(f"{name:10s} | MIA AUC {mia:.3f} | forget NLL {fn:.3f} | holdout NLL {hn:.3f} | retain NLL {rn:.3f}")
    del m; gc.collect(); torch.cuda.empty_cache()

# ---------- unlearning ----------
def cyclic(exs, bs):
    buf = []
    while True:
        if len(buf) < bs:
            idx = list(range(len(exs))); random.shuffle(idx); buf += idx
        take, buf = buf[:bs], buf[bs:]
        yield [exs[i] for i in take]

def run_unlearn(method, lr):
    set_seed()
    model = load_model(target_path, for_training=True)
    model.train()

    ref = load_model(target_path, for_training=False) if method == "npo" else None
    if ref:
        for p in ref.parameters(): p.requires_grad_(False)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    scaler = torch.amp.GradScaler("cuda")
    fstream, rstream = cyclic(forget_e, UN_MICRO_BS), cyclic(retain_e, UN_MICRO_BS)

    print(f"Running {method.upper()} lr={lr:g}...", end=" ", flush=True)
    opt.zero_grad(set_to_none=True)

    for step in range(1, UN_STEPS + 1):
        fb = next(fstream)

        with torch.autocast("cuda"):
            nll, mask = token_nll(model, *collate(fb))
            fn = (nll * mask).sum() / mask.sum()

            if method == "ga":
                loss = -fn
            elif method == "graddiff":
                rb = next(rstream); rn, rm = token_nll(model, *collate(rb))
                loss = -fn + (rn * rm).sum() / rm.sum()
            elif method == "npo":
                logp = -(nll * mask).sum(1)
                with torch.no_grad():
                    rn, rm = token_nll(ref, *collate(fb))
                    logp_ref = -(rn * rm).sum(1)
                loss = -(2.0 / NPO_BETA) * F.logsigmoid(-NPO_BETA * (logp - logp_ref)).mean()

            loss = loss / UN_ACCUM_STEPS

        scaler.scale(loss).backward()

        if step % UN_ACCUM_STEPS == 0 or step == UN_STEPS:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)

    mia, fn, hn = eval_mia(model)
    rn = mean_nll(model, retain_e)
    print(f"Done.\n  -> MIA AUC {mia:.3f} | forget NLL {fn:.3f} | holdout NLL {hn:.3f} | retain NLL {rn:.3f}")

    del model, ref, opt; gc.collect(); torch.cuda.empty_cache()

print("\n--- Unlearning Sweep ---")
for m in METHODS:
    for lr in UN_LRS:
        run_unlearn(m, lr)

print("\nFeasibility Check Complete.")

device: cuda | gpu: Tesla T4
Loading and slicing dataset...


splits: forget 50 | holdout 50 | retain 150


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!



--- Fine-tuning Target (Mamba) ---


Loading weights:   0%|          | 0/242 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/242 [00:00<?, ?it/s]

Running NPO lr=0.0001... Done.
  -> MIA AUC 0.996 | forget NLL 0.522 | holdout NLL 4.779 | retain NLL 0.271


Loading weights:   0%|          | 0/242 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/242 [00:00<?, ?it/s]

Running NPO lr=0.0003... Done.
  -> MIA AUC 0.970 | forget NLL 1.198 | holdout NLL 4.989 | retain NLL 0.510

Feasibility Check Complete.
